## 26. تحلیل متن آگهی

حداقل عبارت‌های زیر بررسی شوند:

- نوساز
- کلیدنخورده
- فوری
- معاوضه
- زیر قیمت

مراحل مورد انتظار:

1. نرمال‌سازی متن فارسی
2. جست‌وجوی چند شکل نوشتاری
3. استخراج Featureهای باینری یا شمارشی
4. بررسی فراوانی
5. مقایسه قیمت خام
6. مقایسه کنترل‌شده برای واحدهای مشابه
7. بررسی Precision روی نمونه دستی

نتیجه باید مشخص کند که عبارت‌ها:

- با قیمت پایین‌تر یا بالاتر همراه‌اند؛
- یا پس از کنترل ویژگی‌ها رابطه قابل توجهی ندارند؛
- یا داده برای نتیجه‌گیری کافی نیست.

نرمال سازی متن فارسی در بخش تمیز کردن داده انجام شده است

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_feather("../Outputs/21_df.feather")

<div dir="rtl" align="right">

 استخراج ویژگی‌های باینری از متن

</div>

In [3]:
def extract_bi_ready_features(df: pd.DataFrame) -> pd.DataFrame:

    keyword_columns = {
        'has_newly_built': ['نوساز', 'نوساخته', 'تازه ساز'],
        'has_never_lived': ['کلیدنخورده', 'کلید نخورده', 'صفر کلید'],
        'has_urgent': ['فوری', 'فورا', 'فوریت'],
        'has_swap': ['معاوضه', 'مبادله', 'تعویض'],
        'has_below_market': ['زیر قیمت', 'مناسب قیمت', 'قیمت عالی'],
        'has_discount': ['تخفیف', 'تخفیف ویژه'],
        # 'has_premium': ['ویژه', 'لوکس', 'لاکچری', 'مدرن', 'شیک'],
        # 'has_high_quality': ['فول', 'کامل', 'تمیز', 'مرتب', 'عالی']
    }
    
    # ترکیب متن
    df['search_text'] = df['title'].fillna('') + ' ' + df['description'].fillna('')
    
    # ایجاد ستون‌های Boolean
    for col_name, patterns in keyword_columns.items():
        pattern = '|'.join(patterns)
        df[col_name] = df['search_text'].str.contains(pattern, case=False, na=False)
    
    # 2. ستون Tag ترکیبی (برای نمایش)
    def create_tags(row):
        tags = []
        for col_name in keyword_columns.keys():
            if row[col_name]:
                tag = col_name.replace('has_', '')
                tags.append(tag)
        return '|'.join(tags) if tags else None
    
    df['keywords'] = df.apply(create_tags, axis=1)
    
    # 3. ستون شمارش
    df['keyword_count'] = df['keywords'].str.split('|').str.len()
    df['keyword_count'] = df['keyword_count'].fillna(0).astype(int)
    
    
    # حذف ستون موقت
    df = df.drop('search_text', axis=1)
    
    return df


# اجرا روی دیتاست
df = extract_bi_ready_features(df)



In [4]:
df.columns

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
       'created_at_month', 'user_type', 'description', 'title', 'rent_mode',
       'rent_value', 'rent_to_single', 'rent_type', 'price_mode',
       'price_value', 'credit_mode', 'credit_value', 'rent_credit_transform',
       'transformable_price', 'transformable_credit', 'transformed_credit',
       'transformable_rent', 'transformed_rent', 'land_size', 'building_size',
       'deed_type', 'has_business_deed', 'floor', 'rooms_count',
       'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator',
       'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt',
       'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas',
       'has_heating_system', 'has_cooling_system', 'has_restroom',
       'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool',
       'has_jacuzzi', 'has_sauna', 'floor_material', 'property_type',
       'regular_person_capacity', 'extra_person

<div dir="rtl" align="right">

خلاصه فراوانی کلمات کلیدی

تعداد و درصد آگهی‌هایی که هر کلمه کلیدی در آن‌ها وجود دارد محاسبه و به ترتیب نزولی نمایش داده می‌شود.

</div>

In [5]:
has_columns = ["has_newly_built","has_never_lived","has_urgent","has_swap","has_below_market","has_discount"]


print("\n" + "=" * 50)
print(" جدول خلاصه:")
print("=" * 50)

summary_df = pd.DataFrame({
    'keyword': [col for col in has_columns],
    'count': [df[col].sum() for col in has_columns],
    'percent': [(df[col].sum() / len(df)) * 100 for col in has_columns]
})
summary_df = summary_df.sort_values('count', ascending=False)
print(summary_df.to_string(index=False))



 جدول خلاصه:
         keyword  count  percent
        has_swap  85832 8.583603
    has_discount  70722 7.072532
 has_newly_built  66651 6.665413
      has_urgent  43100 4.310203
 has_never_lived  39887 3.988887
has_below_market  37004 3.700574


<div dir="rtl" align="right">
تحلیل قیمت بر اساس رژیم معاملاتی
</div>

In [6]:
target_regimes = ['sell', 'mortgage_only', 'rent_only']
has_columns = ["has_newly_built","has_never_lived","has_urgent","has_swap","has_below_market","has_discount"]

print("=" * 80)
print("تحلیل کلمات کلیدی برای فروش، رهن کامل، اجاره کامل")
print("=" * 80)

# ==============================================
# 1. رژیم فروش (sell)
# ==============================================

if 'sell' in df['price_regime'].unique():
    print("\n" + "="*80)
    print(" رژیم: فروش (sell)")
    print("="*80)
    
    sell_df = df[(df['price_regime'] == 'sell') & (df['sale_price'] > 0)]
    
    if len(sell_df) > 0:
        print(f"\nتعداد کل آگهی‌های فروش: {len(sell_df):,}")
        print(f"میانگین قیمت کل: {sell_df['sale_price'].mean():,.0f}")
        print(f"میانه قیمت کل: {sell_df['sale_price'].median():,.0f}")
        print("\n" + "-"*80)
        print(f"{'Keyword':<15} {'Count':<10} {'Average':<18} {'Median':<18} {'Difference':<18} {'Result'}")
        print("-"*80)
        
        for col in has_columns:
            keyword_df = sell_df[sell_df[col] == True]
            if len(keyword_df) >= 10:
                mean_price = keyword_df['sale_price'].mean()
                median_price = keyword_df['sale_price'].median()
                count = len(keyword_df)
                diff = mean_price - sell_df['sale_price'].mean()
                diff_pct = (diff / sell_df['sale_price'].mean()) * 100
                
                if diff_pct > 5:
                    result = " بالاتر"
                elif diff_pct < -5:
                    result = " پایین‌تر"
                else:
                    result = " مشابه"
                
                print(f"{col.replace('has_',''):<15} {count:>8,}   {mean_price:>15,.0f}   {median_price:>15,.0f}   {diff:>13,.0f} ({diff_pct:>5.1f}%)   {result}")

# ==============================================
# 2. رژیم رهن کامل (mortgage_only)
# ==============================================

if 'mortgage_only' in df['price_regime'].unique():
    print("\n" + "="*80)
    print(" رژیم: رهن کامل (mortgage_only)")
    print("="*80)
    
    mortgage_df = df[(df['price_regime'] == 'mortgage_only') & (df['deposit_amount'] > 0)]
    
    if len(mortgage_df) > 0:
        print(f"\nتعداد کل آگهی‌های رهن کامل: {len(mortgage_df):,}")
        print(f"میانگین مبلغ رهن کل: {mortgage_df['deposit_amount'].mean():,.0f}")
        print(f"میانه مبلغ رهن کل: {mortgage_df['deposit_amount'].median():,.0f}")
        print("\n" + "-"*80)
        print(f"{'Keyword':<15} {'Count':<10} {'Average':<18} {'Median':<18} {'Difference':<18} {'Result'}")
        print("-"*80)
        
        for col in has_columns:
            keyword_df = mortgage_df[mortgage_df[col] == True]
            if len(keyword_df) >= 10:
                mean_price = keyword_df['deposit_amount'].mean()
                median_price = keyword_df['deposit_amount'].median()
                count = len(keyword_df)
                diff = mean_price - mortgage_df['deposit_amount'].mean()
                diff_pct = (diff / mortgage_df['deposit_amount'].mean()) * 100
                
                if diff_pct > 5:
                    result = " بالاتر"
                elif diff_pct < -5:
                    result = " پایین‌تر"
                else:
                    result = " مشابه"
                
                print(f"{col.replace('has_',''):<15} {count:>8,}   {mean_price:>15,.0f}   {median_price:>15,.0f}   {diff:>13,.0f} ({diff_pct:>5.1f}%)   {result}")

# ==============================================
# 3. رژیم اجاره کامل (rent_only)
# ==============================================

if 'rent_only' in df['price_regime'].unique():
    print("\n" + "="*80)
    print(" رژیم: اجاره کامل (rent_only)")
    print("="*80)
    
    rent_df = df[(df['price_regime'] == 'rent_only') & (df['monthly_rent'] > 0)]
    
    if len(rent_df) > 0:
        print(f"\nتعداد کل آگهی‌های اجاره کامل: {len(rent_df):,}")
        print(f"میانگین اجاره ماهانه کل: {rent_df['monthly_rent'].mean():,.0f}")
        print(f"میانه اجاره ماهانه کل: {rent_df['monthly_rent'].median():,.0f}")
        print("\n" + "-"*80)
        print(f"{'Keyword':<15} {'Count':<10} {'Average':<18} {'Median':<18} {'Difference':<18} {'Result'}")
        print("-"*80)
        
        for col in has_columns:
            keyword_df = rent_df[rent_df[col] == True]
            if len(keyword_df) >= 10:
                mean_price = keyword_df['monthly_rent'].mean()
                median_price = keyword_df['monthly_rent'].median()
                count = len(keyword_df)
                diff = mean_price - rent_df['monthly_rent'].mean()
                diff_pct = (diff / rent_df['monthly_rent'].mean()) * 100
                
                if diff_pct > 5:
                    result = " بالاتر"
                elif diff_pct < -5:
                    result = " پایین‌تر"
                else:
                    result = "مشابه"
                
                print(f"{col.replace('has_',''):<15} {count:>8,}   {mean_price:>15,.0f}   {median_price:>15,.0f}   {diff:>13,.0f} ({diff_pct:>5.1f}%)   {result}")

print("\n" + "="*80)
print("تحلیل کامل شد")

تحلیل کلمات کلیدی برای فروش، رهن کامل، اجاره کامل

 رژیم: فروش (sell)

تعداد کل آگهی‌های فروش: 565,190
میانگین قیمت کل: 17,447,858,310
میانه قیمت کل: 2,850,000,000

--------------------------------------------------------------------------------
Keyword         Count      Average            Median             Difference         Result
--------------------------------------------------------------------------------
newly_built       40,410    26,825,713,419     4,300,000,000   9,377,855,110 ( 53.7%)    بالاتر
never_lived       25,634    28,124,298,650     5,300,000,000   10,676,440,340 ( 61.2%)    بالاتر
urgent            29,631    13,821,853,156     2,580,000,000   -3,626,005,153 (-20.8%)    پایین‌تر
swap              76,823    15,965,050,623     2,400,000,000   -1,482,807,687 ( -8.5%)    پایین‌تر
below_market      31,106    20,808,859,692     3,550,000,000   3,361,001,382 ( 19.3%)    بالاتر
discount          54,309    10,685,972,046     2,300,000,000   -6,761,886,264 (-38.8%)    پایین

In [7]:
df.to_feather("../Outputs/26_df.feather")